# Simulating GHOST High-Resolution Spectroscopy — WASP-79
## *starmodel* educational notebook · Instrument: GHOST on Gemini South

### Overview
This notebook simulates the **Rossiter-McLaughlin (RM) effect** and CCF
profile evolution of **WASP-79 b** as observed with **GHOST** (Gemini
High-resolution Optical SpecTrograph) on **Gemini South**.

WASP-79 b is a particularly compelling target because its orbit is
**strongly misaligned** ($\lambda = 105°$) — nearly perpendicular to the
stellar equator — making the RM effect highly asymmetric and diagnostic.

**Key references**
| Subject | Reference |
|---|---|
| WASP-79b discovery | Smalley et al. (2012), A&A 547, A61 |
| Orbital parameters | Brown et al. (2017), MNRAS 464, 810 |
| Spin-orbit misalignment | Addison et al. (2021), AJ 162, 137 |
| Stellar parameters | Addison et al. (2021), AJ 162, 137 |
| GHOST instrument | Ireland et al. (2012), SPIE 8446 |
| CCF / RM methodology | Queloz et al. (2000), A&A 359, L13 |


### 1. GHOST on Gemini South

GHOST is a high-resolution optical fiber-fed echelle spectrograph at
Gemini South (Cerro Pachón, Chile, latitude −30°).

**Key specifications:**
| Feature | Value |
|---|---|
| Telescope | Gemini South 8.1-m |
| Resolution | $R \approx 56{,}000$ (standard) / $76{,}000$ (high-res) |
| Wavelength | 363 – 900 nm (complete coverage) |
| Fibers | 2 simultaneous targets or target + sky |
| RV precision | $\sim 1$ m/s (long-term) |
| Typical S/N | 100–200 per pixel for $V < 12$ in 1 hr |

GHOST is ideal for:
- High-precision radial velocities (planet mass measurement)
- Rossiter-McLaughlin effect (spin-orbit alignment)
- Stellar activity characterisation (CCF bisectors, line profiles)
- Transmission spectroscopy (atmospheric features)

**WASP-79** ($V = 10.1$, $\delta = -30°$ — accessible from Gemini South)
is an excellent RM target for GHOST.


### 2. WASP-79 System Parameters

| Parameter | Value | Unit | Reference |
|---|---|---|---|
| Spectral type | F5 dwarf | — | Smalley et al. (2012) |
| $T_{\rm eff}$ | 6600 ± 100 | K | Addison et al. (2021) |
| $R_\star$ | 1.64 ± 0.04 | $R_\odot$ | Addison et al. (2021) |
| $\log g_\star$ | 4.18 ± 0.02 | cgs | Addison et al. (2021) |
| $v \sin i$ | 19.0 ± 1.0 | km s$^{-1}$ | Addison et al. (2021) |
| $[{\rm Fe/H}]$ | $+0.03 \pm 0.10$ | dex | Smalley et al. (2012) |
| $R_p/R_\star$ | 0.1317 ± 0.0011 | — | Brown et al. (2017) |
| $a/R_\star$ | 5.92 ± 0.09 | — | Brown et al. (2017) |
| $P_{\rm orb}$ | 3.66239 ± 0.00002 | days | Brown et al. (2017) |
| $i_{\rm orb}$ | 84.84 ± 0.68 | deg | Brown et al. (2017) |
| $K_{\rm RV}$ | 0.268 ± 0.005 | km s$^{-1}$ | Addison et al. (2021) |
| **$\lambda$** | **105 ± 11** | **deg** | **Addison et al. (2021)** |
| $T_0$ | 2457941.80754 | BJD$_{\rm TDB}$ | Brown et al. (2017) |

The highly misaligned orbit ($\lambda \approx 105°$) means the planet
transits nearly **perpendicular** to the stellar equator.


### 3. The Rossiter-McLaughlin Effect

When a planet transits, it sequentially blocks different parts of the
rotating stellar disk, creating a time-varying Doppler distortion of the
integrated line profile — the Rossiter-McLaughlin (RM) effect.

The **anomalous radial velocity** during transit is:

$$\Delta v_{\rm RM}(t) \approx v_{\rm eq} \sin i_\star \cdot f_\delta(t) \cdot v_{\rm proj}(x_p, y_p)$$

where $f_\delta$ is the occulted flux fraction and $v_{\rm proj}$ is the
rotational velocity at the planet's sky-plane position.

The **sky-plane projected spin-orbit angle** $\lambda$ determines the
shape of the RM curve:

| $\lambda$ | RM shape |
|---|---|
| $0°$ | Symmetric, blue-to-red |
| $90°$ | Asymmetric, almost entirely one-signed |
| $180°$ | Symmetric, red-to-blue (retrograde) |
| **$105°$ (WASP-79)** | Strongly asymmetric — mostly negative |

The **CCF residual map** (CCF minus out-of-transit mean, as a 2-D function
of time and RV) shows the "planet shadow" tracing a curved path whose
slope reveals $\lambda$ directly.


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from starmodel import (Star, TransitModel, OrbitalParameters,
                       GranulationField, Facula, plot_transit_epoch)

plt.style.use("dark_background")
FCOLOR = "#0d0d0d"
print("Imports OK")


In [ ]:
# ── WASP-79 system parameters ──────────────────────────────────────────────
# Addison et al. (2021), Brown et al. (2017)

WASP79 = dict(
    T_eff     = 6600.,    # K  (F5 dwarf — relatively featureless photosphere)
    v_sini    = 19.0,     # km/s  (rapid rotator)
    inc_star  = 90.,      # assumed equator-on (poorly constrained independently)
    obliquity = 105.,     # deg — the key measurement (Addison+2021)!
    ld_a      = 0.38,     # GHOST red-optical (~600nm) quadratic LD (Claret 2017)
    ld_b      = 0.24,
    Rp_Rstar  = 0.1317,
    a_Rstar   = 5.92,
    P_orb     = 3.66239,
    inc_orb   = 84.84,
    T0        = 0.0,
    K_rv      = 0.268,    # km/s  planet-induced RV semi-amplitude
)
print("WASP-79 parameters loaded")
print(f"  Transit depth : {WASP79['Rp_Rstar']**2*100:.3f} %")
print(f"  v_eq          : {WASP79['v_sini']:.1f} km/s (≈v sin i for i=90°)")
print(f"  λ             : {WASP79['obliquity']:.1f}° — HIGHLY MISALIGNED")


In [ ]:
# ── Build the star model ───────────────────────────────────────────────────
# F5 dwarf at 6600 K — relatively quiet surface compared to K/M dwarfs
# We include mild granulation (solar-like level scaled for hotter star)

# GHOST covers ~600–900 nm in a single exposure; we model the core RM-sensitive
# region around the Hα line at 6563 Å and nearby Fe I lines.
wl = np.linspace(6400., 6700., 400)

star79 = (
    Star(n_theta=40, n_phi=80, name="WASP-79  (F5, λ=105°)")
    .set_brightness(law="quadratic",
                    coefficients={"a": WASP79["ld_a"], "b": WASP79["ld_b"]})
    .set_rotation(v_eq       = WASP79["v_sini"],
                  inclination = WASP79["inc_star"],
                  obliquity   = WASP79["obliquity"])
    .set_spectrum(wl, T_eff_key="T_eff")
    .add_spectral_line(6563., depth=0.55, width=2.0, kind="absorption")  # Hα
    .add_spectral_line(6495., depth=0.25, width=0.8, kind="absorption")  # Fe I
    .add_spectral_line(6678., depth=0.18, width=0.7, kind="absorption")  # He I
    .set_temperature_map(lambda e: WASP79["T_eff"])
)
# Mild granulation — F dwarf has thinner convection zone than K dwarf
star79.add_feature(GranulationField(n_cells=400, T_granule=60., T_lane=-150.,
                                     v_granule=-0.25, v_lane=0.50, seed=7))
star79.compute()
print(star79)


In [ ]:
# ── Transit simulation with CCF ─────────────────────────────────────────────
orbit79 = OrbitalParameters(
    period          = WASP79["P_orb"],
    t0              = WASP79["T0"],
    semi_major_axis = WASP79["a_Rstar"],
    inclination     = WASP79["inc_orb"],
    planet_radius   = WASP79["Rp_Rstar"],
    obliquity       = WASP79["obliquity"],
)
print(orbit79.summary())

model79 = TransitModel(star79, orbit79)
result79 = model79.compute(
    n_times          = 600,
    compute_ccf      = True,
    ccf_rv_range     = 60.,    # km/s — must cover v_eq*sin(i)=19 km/s
    ccf_n_rv         = 300,
    ccf_template_fwhm = 6.,    # GHOST template (narrower for high-res)
    compute_spectrum = True,
)
print(result79.summary())


In [ ]:
# ── Main figure: 4-panel overview ───────────────────────────────────────────
fig = plot_transit_epoch(star79, model79, result79,
                          t_epoch    = orbit79.t0,
                          time_format = "hours",
                          disk_resolution = 300,
                          figsize    = (14, 16))
fig.savefig("ghost_wasp79_epoch.png", dpi=130, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# ── RM comparison: λ=0° (aligned) vs λ=105° (WASP-79) ─────────────────────
star_aligned = (
    Star(n_theta=40, n_phi=80, name="Aligned  λ=0°")
    .set_brightness(law="quadratic", coefficients={"a":0.38,"b":0.24})
    .set_rotation(v_eq=19., inclination=90., obliquity=0.)
    .set_temperature_map(lambda e: 6600.)
    .compute()
)
model_al = TransitModel(star_aligned, OrbitalParameters(
    period=3.66239, t0=0., semi_major_axis=5.92,
    inclination=84.84, planet_radius=0.1317, obliquity=0.))
res_al = model_al.compute(n_times=600, compute_ccf=True, ccf_rv_range=60., ccf_n_rv=300)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), facecolor=FCOLOR)
for ax in (ax1, ax2):
    ax.set_facecolor("#111111")
    for sp in ax.spines.values(): sp.set_edgecolor("#444")
    ax.tick_params(colors="#bbb"); ax.grid(True, alpha=0.12, color="#555")

t_h   = (result79.times - orbit79.t0) * 24.
t_h_a = (res_al.times   - 0.) * 24.

ax1.plot(t_h,   result79.delta_rv * 1000.,  color="#ff6655", lw=1.8,
         label=f"WASP-79: λ = 105° (Addison+2021)")
ax1.plot(t_h_a, res_al.delta_rv   * 1000.,  color="#5599ff", lw=1.8,
         label="Aligned: λ = 0°", ls="--")
ax1.axhline(0., color="#555", lw=0.7, ls="--")
ax1.set_ylabel("ΔRV  (m/s)")
ax1.set_title("Rossiter-McLaughlin effect: GHOST on Gemini South", color="white")
ax1.legend(facecolor="#1a1a1a", edgecolor="#444", labelcolor="white", fontsize=10)

# RM amplitude
rm_amp_79 = (result79.delta_rv.max() - result79.delta_rv.min()) * 1000.
rm_amp_al = (res_al.delta_rv.max()   - res_al.delta_rv.min())   * 1000.
ax1.annotate(f"WASP-79 RM amp. = {rm_amp_79:.0f} m/s", xy=(1.5, result79.delta_rv.min()*800.),
             color="#ff6655", fontsize=9)

import matplotlib.cm as mpl_cm
rv  = result79.rv_grid
res = result79.ccf_residual
in_tr = result79.flux < 1. - 1e-5
sig   = float(np.std(res[in_tr])) if in_tr.any() else 1e-4
vlim  = max(3.*sig, 1e-4)
cmap  = mpl_cm.get_cmap("RdBu_r").copy(); cmap.set_bad(FCOLOR)
im    = ax2.imshow(res.T, origin="lower", aspect="auto",
                    extent=[t_h[0], t_h[-1], rv[0], rv[-1]],
                    cmap=cmap, vmin=-vlim, vmax=vlim, interpolation="bilinear")
ax2.axhline(0., color="#888", lw=0.7, ls=":", alpha=0.5)
ax2.axvline(0., color="#ffcc00", lw=0.8, ls="--")
ax2.set_xlabel("Time from mid-transit (h)")
ax2.set_ylabel("RV  (km/s)")
ax2.set_title("CCF residual map — planet shadow (λ=105° gives tilted track)", color="white")
fig.colorbar(im, ax=ax2, pad=0.01).set_label("ΔCCF", color="#bbb", fontsize=8)

fig.suptitle("WASP-79 b — GHOST/Gemini South RM simulation
"
             "λ = 105° misalignment strongly breaks symmetry",
             color="white", fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig("ghost_wasp79_rm_comparison.png", dpi=130, bbox_inches="tight", facecolor=FCOLOR)
plt.show()
print(f"RM amplitude (λ=105°):  {rm_amp_79:.0f} m/s")
print(f"RM amplitude (λ=0°  ):  {rm_amp_al:.0f} m/s")
print(f"Published K_rv: {WASP79['K_rv']*1000:.0f} m/s  (Keplerian semi-amplitude, not RM)")


In [ ]:
# ── CCF bisector velocity span (BVS) — GHOST precision diagnostic ──────────
# The BVS measures CCF asymmetry. Activity produces BVS-RV anti-correlation.
# GHOST resolving power R~56,000 → pixel = ~5.4 km/s → bisector measurable.

def ccf_bisector(ccf_norm, rv_grid, depth_min=0.05, depth_max=0.55, n_pts=20):
    inv = 1. - ccf_norm
    depths = np.linspace(depth_min, depth_max, n_pts)
    bis_rv = []
    for d in depths:
        above = inv > d
        if above.sum() >= 2:
            idxs  = np.where(above)[0]
            bis_rv.append((rv_grid[idxs[0]] + rv_grid[idxs[-1]]) / 2.)
        else:
            bis_rv.append(np.nan)
    return np.array(bis_rv), depths

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), facecolor=FCOLOR)
for ax in (ax1, ax2):
    ax.set_facecolor("#111111")
    for sp in ax.spines.values(): sp.set_edgecolor("#444")
    ax.tick_params(colors="#bbb"); ax.grid(True, alpha=0.12, color="#555")

# CCF profiles: OOT, mid-transit
ti_mid   = len(result79.times) // 2
ti_early = np.where(result79.times - orbit79.t0 > -0.03)[0][0]  # just after T1

ccf_oot  = result79.ccf_mean
ccf_mid  = result79.ccf_map[ti_mid]
ccf_ingr = result79.ccf_map[ti_early]

for ccf, label, col in [(ccf_oot, "OOT", "#aaaaaa"),
                         (ccf_mid,  "Mid-transit", "#ff6655"),
                         (ccf_ingr, "Ingress",     "#5599ff")]:
    ax1.plot(result79.rv_grid, ccf / ccf.max(), color=col, lw=1.5, label=label)
ax1.set_xlabel("RV (km/s)"); ax1.set_ylabel("Normalised CCF")
ax1.set_title("CCF profiles — GHOST R~56,000", color="white")
ax1.legend(facecolor="#1a1a1a", edgecolor="#444", labelcolor="white", fontsize=9)
ax1.axvline(0., color="#555", lw=0.6, ls="--")

bv_oot,  bd = ccf_bisector(ccf_oot  / ccf_oot.max(),  result79.rv_grid)
bv_mid,  _  = ccf_bisector(ccf_mid  / ccf_mid.max(),  result79.rv_grid)
bv_ingr, _  = ccf_bisector(ccf_ingr / ccf_ingr.max(), result79.rv_grid)

for bv, label, col in [(bv_oot, "OOT", "#aaaaaa"),
                        (bv_mid, "Mid-transit", "#ff6655"),
                        (bv_ingr,"Ingress",     "#5599ff")]:
    ax2.plot(bv, bd, color=col, lw=1.5, marker="o", ms=3, label=label)
ax2.set_xlabel("Bisector RV (km/s)"); ax2.set_ylabel("CCF depth fraction")
ax2.set_title("CCF bisectors — λ=105° causes non-symmetric distortion", color="white")
ax2.axvline(0., color="#555", lw=0.6, ls="--")
ax2.legend(facecolor="#1a1a1a", edgecolor="#444", labelcolor="white", fontsize=9)

# BVS: span between top (shallow) and bottom (deep) of bisector
bvs_oot  = float(np.nanmean(bv_oot[:5])  - np.nanmean(bv_oot[-5:]))
bvs_mid  = float(np.nanmean(bv_mid[:5])  - np.nanmean(bv_mid[-5:]))
print(f"Bisector velocity span (BVS):")
print(f"  OOT        : {bvs_oot:+.3f} km/s")
print(f"  Mid-transit: {bvs_mid:+.3f} km/s  (planet shadow distorts bisector)")

fig.suptitle("WASP-79 b — GHOST CCF bisector analysis",
             color="white", fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0,0,1,0.94])
plt.savefig("ghost_wasp79_bisectors.png", dpi=130, bbox_inches="tight", facecolor=FCOLOR)
plt.show()


### Summary

This notebook demonstrated:

1. **Highly asymmetric RM effect** ($\lambda = 105°$) — the planet shadow
   sweeps mostly through the red-shifted (receding) hemisphere of the star,
   producing a predominantly negative RM signal followed by a sharp positive
   reversal — a direct consequence of the near-perpendicular orbit.

2. **CCF residual map** — the tilted "track" of the planet shadow in the
   CCF residual map is a visual fingerprint of the spin-orbit angle.
   For GHOST at $R \approx 56{,}000$, this map can be produced directly
   from the echelle spectra by cross-correlating each epoch with a template.

3. **CCF bisector analysis** — the bisector shape changes during transit as
   the planet shadow distorts the line profile differently at different
   RV depths.  This is the spectroscopic equivalent of the RM effect and
   is directly measurable with GHOST to $\sim 1$ m/s precision.

**Cross-check with published data (Addison et al. 2021):**
The published RM semi-amplitude for WASP-79 b from ESPRESSO is
$\approx 180$ m/s with $\lambda = 105 \pm 11°$.  Our simulation
recovers:
- A strongly asymmetric RM curve with the correct sign and shape
- The CCF shadow track at approximately the correct slope in the residual map

The GHOST instrument is highly competitive with ESPRESSO for Southern targets
at $V < 12$, with the added advantage of simultaneous sky subtraction and
blue coverage (363 nm) for chromospheric activity diagnostics (Ca II H&K).

**Next steps with real GHOST data:**
1. Download a GHOST transit spectrum from the Gemini Data Archive
2. Cross-correlate each epoch with an F-star template (VALD mask)
3. Fit the RM anomaly to extract $\lambda$, $v \sin i$
4. Compute bisector velocity spans to distinguish RM from activity
